In [52]:
from markitdown import MarkItDown
from langchain_core.documents import Document
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from dotenv import load_dotenv
import os
import re
load_dotenv()

True

In [37]:
client = OpenAI()
md = MarkItDown(llm_client=client, llm_model="gpt-4o")
result = md.convert("../assets/knowledgebase/Calibration CAD CAM.pptx")
print(result.text_content)

<!-- Slide number: 1 -->
# CAD / CAMcALIBRATION
MUSC  JBECDM
Department of Reconstructive & Rehabilitation Sciences
Division of Digital Dentistry

![White Jigsaw puzzle on yellow color background](Picture3.jpg)

<!-- Slide number: 2 -->
# Synopsis of Procedure
CAD/CAM Restoration of a single anterior or posterior tooth utilizing e-max and either the Planmeca Fit system or the CEREC system.
3 hours to complete the procedure.
Patient should pay prior to being brought back to the clinic for treatment.
If significant decay exists in the tooth, such as might put the nerve at risk of an irreversible pulpitis, the decay should be removed and a build up performed at a procedure prior to the CAD/CAM visit.  The final crown appointment should be delayed at least 3 weeks to see that the tooth remains asymptomatic.
The expectation is that the preparation be complete 1:15 after the appointment begins.

<!-- Slide number: 3 -->
# Armamentarium
Digital scanner
Depth cut bur
016 KR round ended should

# Create Documents Chunk

In [43]:
root = '../assets/knowledgebase/'
files = [pptx_file for pptx_file in os.listdir(root) if pptx_file[-5:] == '.pptx']
slides = {}
for file in files:
    result = md.convert(root + file).text_content
    content = re.split(r'<!-- Slide number: \d+ -->\n', result)
    content = [cont.strip() for cont in content if cont.strip()]
    slides[file] = content

In [44]:
slides.keys()

dict_keys(['Calibration CAD CAM.pptx', 'CAMBRA Interventions.pptx', 'Adhesive Dentistry Calibration Module.pptx'])

In [45]:
from uuid import uuid4
docs = []
for key, val in slides.items():
    docs_bykey = [
        Document(metadata={"file": key, "slide_number": i + 1}, page_content=content, id=uuid4())
        for i, content in enumerate(val)
    ]
    docs += docs_bykey

print(len(docs))

89


In [51]:
docs[:2]

[Document(id='4f50e87b-154f-452d-bb69-e97286eac470', metadata={'file': 'Calibration CAD CAM.pptx', 'slide_number': 1}, page_content='# CAD / CAM\x0bcALIBRATION\nMUSC  JBECDM\nDepartment of Reconstructive &\xa0Rehabilitation Sciences\nDivision of Digital Dentistry\n\n![White Jigsaw puzzle on yellow color background](Picture3.jpg)'),
 Document(id='480dca2d-a65a-4d2f-b5ac-0f36a9f0bf3e', metadata={'file': 'Calibration CAD CAM.pptx', 'slide_number': 2}, page_content='# Synopsis of Procedure\nCAD/CAM Restoration of a single anterior or posterior tooth utilizing e-max and either the Planmeca Fit system or the CEREC system.\n3 hours to complete the procedure.\nPatient should pay prior to being brought back to the clinic for treatment.\nIf significant decay exists in the tooth, such as might put the nerve at risk of an irreversible pulpitis, the decay should be removed and a build up performed at a procedure prior to the CAD/CAM visit.  The final crown appointment should be delayed at least 3 w

In [55]:
import faiss
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

index = faiss.IndexFlatL2(len(embeddings.embed_query("START")))
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [57]:
vector_store.add_documents(documents=docs)

['4f50e87b-154f-452d-bb69-e97286eac470',
 '480dca2d-a65a-4d2f-b5ac-0f36a9f0bf3e',
 '4d787e79-f362-4d19-8b4f-9e02e50d388c',
 'd6a85f67-546e-49ae-bcbf-727feaf16b4e',
 '7c2aa641-4d1c-47a4-b7c8-72c1eaccfe23',
 'e8dcaf42-1aa0-4a37-bb68-fa895e51ce23',
 '16f815e0-2b03-42d9-a091-8ca5a4ca2ba4',
 '8b278702-0257-4f02-9708-4fe6cd64835d',
 '68fd1e87-13a9-4379-93e3-7826453fb056',
 '7661baa6-4c14-423c-af31-d09a81702dca',
 '008bce8c-0098-4fc5-a723-8c6d4a25d209',
 'a82bcea6-946d-4f1c-9820-d853b8139a5f',
 '3e4bbf84-2453-4322-9fa5-119f9425437a',
 'b194cf57-59ab-45e7-b0b2-ecee9e2bb0d9',
 '4b52afc6-f967-419c-92fe-90de2cf88589',
 '3e480956-f68f-45eb-8b54-36e140c53150',
 'a602d2bc-ad00-47bb-aa4a-a99eeb59ddb9',
 '3a4aa34d-5611-498c-9731-c46ef903a416',
 'd57624ee-5161-4655-be54-634ebdd0b6ec',
 '16c609cd-b354-45c2-9ee4-a53c3f471779',
 '708e9886-9b60-451c-941b-4a528949fbb6',
 'a74cc8da-f3ed-4c70-82cf-419b20f932dc',
 '9e36cefc-d963-47e0-9b51-a0d5900a8bb0',
 '3829fadd-ea9d-4be2-9f1a-507ea8ba3343',
 '860c50cd-d556-

In [59]:
vector_store.save_local("../assets/dental_slides/")

loaded_vector_store = FAISS.load_local(
    "../assets/dental_slides/", embeddings, allow_dangerous_deserialization=True
)

In [69]:
Adhesive_retriever = loaded_vector_store.as_retriever(
    search_type="mmr", 
    search_kwargs={
        "filter": {"file": "Adhesive Dentistry Calibration Module.pptx"}
    }
)